# Estudiando las frases

In [5]:
import pandas as pd

In [13]:
df = pd.read_pickle('/home/user/work/Preprocessing/MSKA/Dataset/data/Phoenix-2014T.train.pkl')
rows = []

for k, v in df.items():
    rows.append({
        "id": k,
        "name": v["name"],
        "gloss": v["gloss"],
        "text": v["text"],
        "num_frames": v["num_frames"],
        "keypoint_shape": tuple(v["keypoint"].shape)
    })

df = pd.DataFrame(rows)
df.head()

,id,name,gloss,text,num_frames,keypoint_shape
0,11August_2010_Wednesday_tagesschau-1,11August_2010_Wednesday_tagesschau-1,JETZT WETTER MORGEN DONNERSTAG ZWOELF FEBRUAR,und nun die wettervorhersage für morgen donner...,86,"(86, 136, 3)"
1,25October_2010_Monday_tagesschau-15,25October_2010_Monday_tagesschau-15,ITALIEN IX TIEF DRUCK KOMMEN HEUTE NACHT BERG ...,das tief über italien sorgt dafür dass es an d...,138,"(138, 136, 3)"
2,11August_2010_Wednesday_tagesschau-7,11August_2010_Wednesday_tagesschau-7,WOLKE LOCH SPEZIELL NORDWEST,größere wolkenlücken finden sich vor allem im ...,71,"(71, 136, 3)"
3,11August_2010_Wednesday_tagesschau-9,11August_2010_Wednesday_tagesschau-9,FLUSS HEUTE NACHT SECHS FLUSS SIEBZEHN GRAD,im emsland heute nacht nur neun am oberrhein b...,105,"(105, 136, 3)"
4,11August_2010_Wednesday_tagesschau-13,11August_2010_Wednesday_tagesschau-13,TEMPERATUR BLEIBEN GLEICH,am temperaturniveau ändert sich wenig,48,"(48, 136, 3)"


In [14]:
from transformers import AutoTokenizer

# Cargar el tokenizador
tokenizer_path = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, trust_remote_code=True)

# Función para contar tokens
def count_qwen_tokens(text):
    if not isinstance(text, str):
        return 0
    # Tokenizamos sin truncar para ver la longitud real completa
    return len(tokenizer(text, truncation=False)['input_ids'])

print("Calculando longitudes de tokens...")

# Aplicar la función a las columnas de Texto y Glosas
df['text_tokens'] = df['text'].apply(count_qwen_tokens)


# Mostrar las estadísticas con percentiles
print("\n" + "="*50)
print("📊 ESTADÍSTICAS DE TOKENS DE TEXTO (Alemán)")
print("="*50)
print(df['text_tokens'].describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]))

# Ver la relación Fotogramas vs Tokens
df['ratio_frames_text'] = df['num_frames'] / df['text_tokens']
print("\n" + "="*50)
print("RATIO: Fotogramas de video por cada Token de texto")
print("="*50)
print(df['ratio_frames_text'].describe())

Calculando longitudes de tokens...

📊 ESTADÍSTICAS DE TOKENS DE TEXTO (Alemán)
count    7096.000000
mean       23.590614
std         9.722534
min         3.000000
50%        22.000000
75%        29.000000
90%        37.000000
95%        41.000000
99%        51.000000
max        78.000000
Name: text_tokens, dtype: float64

RATIO: Fotogramas de video por cada Token de texto
count    7096.000000
mean        5.028114
std         1.179292
min         0.551724
25%         4.318182
50%         4.951800
75%         5.619048
max        18.800000
Name: ratio_frames_text, dtype: float64


# Estudiando salidas de S2G

In [7]:
import torch
import yaml
import random
from torch.utils.data import DataLoader
import numpy as np

from Recognition.Tokenizer import GlossTokenizer_S2G

from slm.S2T_Dataset import S2T_Dataset
from slm.model_slm import SignLanguageModel

# Simular los argumentos de terminal 
class DummyArgs:
    def __init__(self):
        # Entrenamiento básico
        self.batch_size = 8
        self.epochs = 40
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.seed = 0
        self.num_workers = 4
        self.config = 'configs/abalation/SLR_Jdropout.yaml'

        # Checkpoints
        self.finetune = ''
        self.resume = '/home/user/work/data/outputs_final/SLR_Jdropout/best_checkpoint.pth'
        self.start_epoch = 0

        # Evaluación
        self.eval = True

        # Memoria
        self.pin_mem = True

        # Weights & Biases
        self.entity = None
        self.project = ''
        self.name = None

        # SLM
        self.slm = False

args = DummyArgs()
device = torch.device(args.device)

# Cargar tu configuración YAML
config_path = args.config
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [2]:
config['gloss']


{'gloss2id_file': 'data/gloss2ids.pkl'}

In [8]:
# Inicializar Tokenizador y Dataset (usamos el dev_label_path)
print("Cargando tokenizador y datos...")
tokenizer = GlossTokenizer_S2G(config['gloss'])

# Cargamos el test data. NOTA: args.phase = 'test'
test_data = S2T_Dataset(
    path=config['data']['dev_label_path'], 
    tokenizer=tokenizer,
    config=config, 
    args=args, 
    phase='val', 
    training_refurbish=False
)

# El truco: shuffle=True en test para sacar 100 muestras aleatorias
test_dataloader = DataLoader(
    test_data,
    batch_size=8, # Usa un batch pequeño para no saturar la RAM de Jupyter
    num_workers=args.num_workers,
    collate_fn=test_data.collate_fn,
    shuffle=True, 
    pin_memory=args.pin_mem
)

# Inicializar Modelo y cargar Pesos
print("Construyendo modelo...")
model = SignLanguageModel(cfg=config, args=args)
model.to(device)

# --- CARGAR PESOS ---
print(f"Cargando pesos desde {args.resume}...")
checkpoint = torch.load(args.resume, map_location=device)

# Usamos strict=False por si tu checkpoint no tiene todo (ej. le falta Qwen o LoRA)
model.load_state_dict(checkpoint['model'], strict=False)
model.eval() # IMPORTANTE: Modo inferencia para desactivar Dropout/BatchNorm
print("¡Modelo listo!")

Cargando tokenizador y datos...
Construyendo modelo...
Cargando pesos desde /home/user/work/data/outputs_final/SLR_Jdropout/best_checkpoint.pth...
¡Modelo listo!


In [9]:
results = []
beam_size = config['testing']['recognition'].get('beam_size', 5)

print("Iniciando extracción de predicciones...")

with torch.no_grad():
    for batch in test_dataloader:
        # El forward pass de tu modelo. Se encarga de mover cosas a CUDA internamente.
        output = model(batch)
        
        # Buscar los logits de las glosas (tu modelo los saca con ese nombre)
        gls_logits = None
        for k, v in output.items():
            if 'gloss_logits' in k:
                gls_logits = v
                break
                
        if gls_logits is None:
            raise ValueError("No se encontraron los gloss_logits en el output del modelo.")

        # Decodificación CTC usando Beam Search
        ctc_decode_output = model.recognition_network.decode(
            gloss_logits=gls_logits,
            beam_size=beam_size,
            input_lengths=output['input_lengths']
        )
        
        # Convertir los IDs numéricos a strings de Glosas
        batch_pred_gls = tokenizer.convert_ids_to_tokens(ctc_decode_output)

        # Almacenar los resultados comparando con el Ground Truth
        for name, hyp, ref in zip(batch['name'], batch_pred_gls, batch['gloss']):
            hyp_str = ' '.join(hyp).upper()
            ref_str = ref.upper() if isinstance(ref, str) else ' '.join(ref).upper()
            
            results.append({
                'ID': name,
                'Ground_Truth': ref_str,
                'Prediccion': hyp_str
            })
            
        # Parar cuando tengamos 100 muestras
        if len(results) >= 100:
            break

# ---------------------------------------------------------
# Mostrar los Resultados
# ---------------------------------------------------------
print(f"\n--- MOSTRANDO 100 PREDICCIONES S2G ---\n")
for i, res in enumerate(results[:100]):
    print(f"[{i+1}/100] Video ID: {res['ID']}")
    print(f"  GT:   {res['Ground_Truth']}")
    print(f"  PRED: {res['Prediccion']}")
    print("-" * 60)

Iniciando extracción de predicciones...

--- MOSTRANDO 100 PREDICCIONES S2G ---

[1/100] Video ID: 20March_2010_Saturday_heute-7288
  GT:   MORGEN WEST VERBREITEN
  PRED: MORGEN SKANDINAVIEN
------------------------------------------------------------
[2/100] Video ID: 12July_2009_Sunday_tagesschau-2574
  GT:   IX SCHAUER SUEDWEST ABEND GEWITTER KOENNEN
  PRED: IX REGEN BESONDERS SUEDWEST ABEND GEWITTER KOENNEN
------------------------------------------------------------
[3/100] Video ID: 31October_2009_Saturday_tagesschau-7147
  GT:   MOEGLICH HEUTE NACHT FROST GLATT VORSICHT FLUSS MOEGLICH PLUS ACHT
  PRED: MOEGLICH HEUTE NACHT FROST GLATT VORSICHT FLUSS MOEGLICH PLUS ACHT
------------------------------------------------------------
[4/100] Video ID: 26May_2010_Wednesday_heute-7867
  GT:   MORGEN REGEN NORDOST SCHNELL VERSCHWINDEN SCHON SCHAUER
  PRED: MORGEN REGEN NORD NORDOST VERSCHWINDEN SCHAUER
------------------------------------------------------------
[5/100] Video ID: 30Septe